# Tải và đọc text embedding Kaggle trên máy local

Chạy bằng Python kernel trong VS Code/Jupyter trên máy của bạn, không cần GPU, Spark hoặc Docker.
Notebook tải một Dataset embedding đã xử lý, tìm bundle, kiểm tra manifest/checksum/Parquet
và xem vector theo batch. Không chạy lại encoder và chưa nhập dữ liệu vào HDFS.

Bắt đầu với `thoandanh/gold-emb-all-beauty-v2`. Dataset riêng tư cần đăng nhập tài khoản
có quyền truy cập. Chờ Kaggle xử lý Dataset xong trước khi tải. Không ghi API token vào ô code.


In [ ]:
%pip install -q "kagglehub>=1.0.3,<2" "pyarrow>=20,<21" numpy pandas


## 1. Chọn Dataset và nơi lưu

Chọn một ngành mỗi lần. Với Video Games/Industrial, điền handle đúng của Dataset bạn đã tạo.
`DATASET_VERSION=None` tải bản mới nhất; để tái lập thực nghiệm nên ghi số version trên Kaggle.
`LOCAL_DATASET_DIR=None` sẽ tải qua KaggleHub. Nếu đã tải, điền đường dẫn được in ở bước 2
để đọc offline. `BUNDLE_NAME` chỉ cần khi thư mục chứa nhiều bundle.

`DOWNLOAD_ROOT` mặc định ở `outputs/processing/embeddings/kaggle_cache` của project,
được loại khỏi Git. KaggleHub giữ cấu trúc cache/version; dùng đúng đường dẫn trả về thay vì đoán tên thư mục.


In [ ]:
from pathlib import Path
import os, json

CATEGORY = 'All_Beauty'
DATASET_HANDLE = 'thoandanh/gold-emb-all-beauty-v2'
# Các ngành tiếp theo: Video_Games, Industrial_and_Scientific.
# Ví dụ handle nếu bạn đã tạo: thoandanh/gold-emb-video-games-v2
DATASET_VERSION = None  # Ví dụ 1, 2; giữ nguyên một version khi làm thực nghiệm.
LOCAL_DATASET_DIR = None  # Ví dụ r'C:\...\versions\1'; có giá trị thì không download/login.
BUNDLE_NAME = None
AUTHENTICATE = False  # True nếu cần mở hộp đăng nhập KaggleHub cho Dataset riêng tư.
ALLOW_PARTIAL = False  # True chỉ để xem checkpoint chưa đủ; chưa dùng cho Gold/model.
FULL_VECTOR_CHECK = True  # Đọc từng batch để kiểm tra finite, norm, IDs và token/chunk counts.

start = Path.cwd().resolve()
REPO_ROOT = next((p for p in (start, *start.parents) if (p/'processing/gold_contract.json').is_file()), start)
DOWNLOAD_ROOT = REPO_ROOT/'outputs/processing/embeddings/kaggle_cache'
REPORT_ROOT = REPO_ROOT/'outputs/processing/embeddings/validation'
os.environ['KAGGLEHUB_CACHE'] = str(DOWNLOAD_ROOT.resolve())
print('Cache:', DOWNLOAD_ROOT.resolve())


## 2. Tải hoặc mở bản local

Nếu báo chưa đăng nhập/không có quyền, đặt `AUTHENTICATE=True`, chạy lại ô dưới và đăng nhập
bằng lời nhắc của KaggleHub. Không paste token vào notebook. Không bật force_download;
SDK dùng lại cache khi cùng version. Để mở lại hoàn toàn offline, dùng `LOCAL_DATASET_DIR`.


In [ ]:
if LOCAL_DATASET_DIR is not None:
    dataset_root = Path(LOCAL_DATASET_DIR).expanduser().resolve()
else:
    import kagglehub
    if AUTHENTICATE:
        kagglehub.login()
    if DATASET_VERSION is not None and (type(DATASET_VERSION) is not int or DATASET_VERSION < 1):
        raise ValueError('DATASET_VERSION phải là số nguyên dương hoặc None')
    handle = DATASET_HANDLE if DATASET_VERSION is None else f'{DATASET_HANDLE}/versions/{DATASET_VERSION}'
    dataset_root = Path(kagglehub.dataset_download(handle)).resolve()
if not dataset_root.is_dir():
    raise FileNotFoundError(dataset_root)
print('Path to dataset files:', dataset_root)
print('Lần sau có thể đặt LOCAL_DATASET_DIR =', repr(str(dataset_root)))


## 3. Hàm đọc và kiểm tra

Chỉ đọc các shard được manifest liệt kê, bỏ qua pilot/tệp tạm. Kiểm tra SHA256, sidecar,
schema float32 384 chiều và số row. Khi FULL_VECTOR_CHECK bật, quét tuần tự để kiểm tra
vector hữu hạn, L2 norm, hash thứ tự IDs và token/chunk totals. RAM chỉ giữ một batch.
Kiểm tra trùng ID nằm trong từng shard; bước này chưa kiểm tra global uniqueness giữa mọi shard
hoặc chứng minh nội dung vector đúng nghĩa. Đây là kiểm tra artifact local, chưa thay thế G3 import.


In [ ]:
import hashlib, re
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024**2), b''):
            digest.update(block)
    return digest.hexdigest()

def shard_path(bundle, relative):
    bundle = Path(bundle).resolve()
    path = (bundle/relative).resolve()
    if not path.is_relative_to(bundle) or not path.relative_to(bundle).parts[0] == 'texts' or path.suffix != '.parquet':
        raise ValueError('Shard phải nằm trong texts/ của bundle: ' + str(relative))
    return path

def vectors_from_batch(batch, dimension=384):
    column = batch.column(batch.schema.get_field_index('embedding'))
    if column.null_count or column.flatten().null_count:
        raise ValueError('Null embedding/value')
    return column.flatten().to_numpy(zero_copy_only=False).reshape(len(batch), dimension)

def validate_bundle(bundle, expected_encoder, allow_partial=False, full_vector_check=True, export_manifest_path=None):
    bundle = Path(bundle).resolve()
    manifest_path = bundle/'embedding_manifest.json'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    status = manifest.get('status')
    if status != 'complete' and not (allow_partial and status == 'partial'):
        raise ValueError(f'Bundle status={status}; cần complete (hoặc ALLOW_PARTIAL=True để xem partial)')
    signature = manifest['signature']
    for key in ('model', 'revision', 'dimension', 'max_seq_length', 'normalize_embeddings', 'output_dtype', 'long_text_policy'):
        if signature.get(key) != expected_encoder[key]:
            raise ValueError('Encoder contract không khớp: ' + key)
    if signature.get('cleaning_version') != 'gold_text_nfc_html_whitespace_v1':
        raise ValueError('Cleaning version không khớp')
    states = manifest.get('shards', [])
    paths = [s['path'] for s in states]
    if not paths or len(paths) != len(set(paths)):
        raise ValueError('Manifest không có shard hoặc có path trùng')
    expected_rows = sum(s['rows'] for s in states)
    if expected_rows != manifest.get('completed_texts', expected_rows):
        raise ValueError('completed_texts không khớp')
    if status == 'complete' and (expected_rows != manifest['texts'] or len(states) != manifest['total_shards']):
        raise ValueError('Bundle complete chưa đủ row/shard')
    if status == 'partial' and (expected_rows > manifest['texts'] or len(states) > manifest['total_shards']):
        raise ValueError('Partial vượt tổng row/shard')
    export = None
    if export_manifest_path is not None:
        export_manifest_path = Path(export_manifest_path)
        if file_sha256(export_manifest_path) != signature['export_manifest_sha256']:
            raise ValueError('Gold export manifest hash không khớp')
        export = json.loads(export_manifest_path.read_text(encoding='utf-8'))
        if export['status'] != 'complete' or export['gold_path'] != manifest['gold_path'] or export['texts'] != manifest['texts']:
            raise ValueError('Gold export lineage/count không khớp')
        source_entries = {s['path']: s for s in export['files']}
        if status == 'complete' and set(paths) != set(source_entries):
            raise ValueError('Thiếu/thừa shard so với Gold export')
    dimension = expected_encoder['dimension']
    checked_rows = 0
    files = []
    for i, state in enumerate(states, 1):
        path = shard_path(bundle, state['path'])
        if state.get('status') != 'complete' or state.get('signature') != signature:
            raise ValueError('Shard chưa complete hoặc khác signature')
        sidecar = json.loads(path.with_suffix('.json').read_text(encoding='utf-8'))
        if sidecar != {k: v for k, v in state.items() if k != 'path'}:
            raise ValueError('Sidecar khác manifest: ' + state['path'])
        if file_sha256(path) != state['output_sha256']:
            raise ValueError('SHA256 không khớp: ' + state['path'])
        if export is not None:
            source = source_entries.get(state['path'])
            if source is None or source['sha256'] != state['input_sha256'] or source['rows'] != state['rows']:
                raise ValueError('Input shard khác Gold export')
        parquet = pq.ParquetFile(path)
        schema = parquet.schema_arrow
        if set(schema.names) != {'text_id','embedding','encoded_tokens','chunks'}:
            raise ValueError('Schema cột không hợp lệ')
        vector_type = schema.field('embedding').type
        if not pa.types.is_fixed_size_list(vector_type) or vector_type.list_size != dimension or not pa.types.is_float32(vector_type.value_type):
            raise ValueError('Embedding phải là fixed-size float32 đúng dimension')
        if schema.field('text_id').type != pa.string() or schema.field('encoded_tokens').type != pa.int64() or schema.field('chunks').type != pa.int32():
            raise ValueError('Kiểu khóa/token/chunk không hợp lệ')
        if parquet.metadata.num_rows != state['rows']:
            raise ValueError('Parquet row count khác manifest')
        if full_vector_check:
            digest = hashlib.sha256()
            seen = set()  # Bound by one producer shard (normally <=25,000 texts).
            token_total = chunk_total = rows = 0
            for batch in parquet.iter_batches(batch_size=2048):
                if any(batch.column(j).null_count for j in range(batch.num_columns)):
                    raise ValueError('Null input field')
                identifiers = batch.column(batch.schema.get_field_index('text_id')).to_pylist()
                for identifier in identifiers:
                    if not re.fullmatch(r'[0-9a-f]{64}', identifier) or identifier in seen:
                        raise ValueError('text_id sai định dạng/trùng trong shard')
                    seen.add(identifier)
                    digest.update((identifier+chr(10)).encode())
                vectors = vectors_from_batch(batch, dimension)
                if not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=0.002):
                    raise ValueError('Vector không hữu hạn hoặc L2 norm không bằng 1')
                tokens = batch.column(batch.schema.get_field_index('encoded_tokens')).to_numpy()
                chunks = batch.column(batch.schema.get_field_index('chunks')).to_numpy()
                if (tokens < 1).any() or (chunks < 1).any():
                    raise ValueError('Token/chunk count phải dương')
                token_total += int(tokens.sum()); chunk_total += int(chunks.sum()); rows += len(batch)
            if (rows, token_total, chunk_total, digest.hexdigest()) != (state['rows'], state['encoded_tokens'], state['chunks'], state['text_ids_sha256']):
                raise ValueError('Row/token/chunk/key digest khác checkpoint')
        checked_rows += state['rows']
        files.append(str(path))
        if i == 1 or i % 10 == 0 or i == len(states):
            print(f'Checked {i}/{len(states)} shards; {checked_rows:,} texts')
    report = dict(status='passed', bundle=str(bundle), manifest_sha256=file_sha256(manifest_path),
        bundle_status=status, rows=checked_rows, expected_texts=manifest['texts'], shards=len(files),
        dimension=dimension, full_vector_check=full_vector_check, gold_export_matched=export is not None,
        global_key_uniqueness_checked=False, complete_for_next_stage=status=='complete')
    return manifest, report, ds.dataset(files, format='parquet')


## 4. Tìm bundle và đối soát

Notebook tự đối chiếu với export manifest G2 local nếu còn trong project. Nếu không có,
vẫn kiểm tra bundle nhưng báo `gold_export_matched=False`; cần đối chiếu lineage khi import Gold.
Không tự chọn bundle đầu tiên khi có nhiều bản cấu hình khác nhau.


In [ ]:
manifest_paths = sorted(dataset_root.rglob('embedding_manifest.json'))
if not manifest_paths:
    raise FileNotFoundError('Không tìm thấy embedding_manifest.json. Kiểm tra Dataset đã xử lý xong và chọn đúng Dataset output.')
for path in manifest_paths:
    info = json.loads(path.read_text(encoding='utf-8'))
    print(path.parent.name, info.get('status'), info.get('completed_texts'), '/', info.get('texts'))
selected = [p for p in manifest_paths if p.parent.name.startswith(CATEGORY+'_') and (BUNDLE_NAME is None or p.parent.name == BUNDLE_NAME)]
if len(selected) != 1:
    raise ValueError('Cần đúng một bundle của CATEGORY. Nếu có nhiều bản, điền BUNDLE_NAME theo danh sách trên.')
bundle_root = selected[0].parent
manifest_header = json.loads(selected[0].read_text(encoding='utf-8'))
if f'/gold/{CATEGORY.lower()}/' not in manifest_header.get('gold_path', ''):
    raise ValueError('CATEGORY khác Gold lineage trong manifest')

contract_path = REPO_ROOT/'processing/gold_contract.json'
if not contract_path.is_file():
    raise FileNotFoundError('Mở notebook từ project hoặc chỉnh REPO_ROOT tới project để đọc Gold contract.')
expected_encoder = json.loads(contract_path.read_text(encoding='utf-8'))['encoder']
gold_run = Path(manifest_header['gold_path']).parent.name
local_export = REPO_ROOT/'outputs/processing/gold'/gold_run/'kaggle_texts/export_manifest.json'
manifest, validation_report, embedding_dataset = validate_bundle(
    bundle_root, expected_encoder, ALLOW_PARTIAL, FULL_VECTOR_CHECK,
    export_manifest_path=local_export if local_export.is_file() else None,
)
validation_report['dataset_handle_requested'] = DATASET_HANDLE
validation_report['dataset_version_requested'] = DATASET_VERSION
validation_report['dataset_root'] = str(dataset_root)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
report_path = REPORT_ROOT/(bundle_root.name + '_validation.json')
report_path.write_text(json.dumps(validation_report, ensure_ascii=False, indent=2)+chr(10), encoding='utf-8')
print(json.dumps(validation_report, ensure_ascii=False, indent=2))
print('Report:', report_path)


## 5. Xem thử dữ liệu

`text_id` là khóa để join; `embedding` là vector 384 chiều, `encoded_tokens`/`chunks`
là thông tin xử lý text. Dataset kết quả không chứa raw text, item ID, user ID hay nhãn.
Cần refs local/HDFS để ánh xạ item/review sang text_id. Không ghép theo thứ tự row.


In [ ]:
import pandas as pd
preview = embedding_dataset.head(5)
records = preview.to_pylist()
display(pd.DataFrame([dict(text_id=r['text_id'], encoded_tokens=r['encoded_tokens'],
    chunks=r['chunks'], dimension=len(r['embedding']), embedding_first_8=r['embedding'][:8]) for r in records]))
if records:
    vector = np.asarray(records[0]['embedding'], dtype=np.float32)
    print('Vector đầu:', vector.shape, vector.dtype, 'L2 norm:', float(np.linalg.norm(vector)))


## 6. Đọc theo batch để dùng tiếp

Ví dụ dưới chỉ đọc batch đầu. Khi xây bước xử lý tiếp theo, bỏ `break` và xử lý/lưu từng
batch. Không gom tất cả batch vào list hoặc gọi `to_pandas()` trên toàn Dataset nếu chưa tính RAM.
Chỉ bundle complete mới chuyển sang G3; với D2, review history vẫn phải lọc theo prediction_time.


In [ ]:
for batch in embedding_dataset.to_batches(batch_size=2048, columns=['text_id','embedding']):
    text_ids = batch.column(batch.schema.get_field_index('text_id')).to_pylist()
    matrix = vectors_from_batch(batch, expected_encoder['dimension'])
    print('Batch:', len(text_ids), 'IDs; matrix:', matrix.shape, matrix.dtype)
    break

print('Bundle complete:', validation_report['complete_for_next_stage'])
print('Chưa import HDFS; bundle nằm tại:', bundle_root)


Đổi ngành: sửa CATEGORY/DATASET_HANDLE, đặt LOCAL_DATASET_DIR=None và chạy lại từ bước 1.
Mỗi Dataset được lưu riêng trong cache. Dataset private cần quyền truy cập của bạn.

Nguồn: [KaggleHub download và authentication](https://github.com/Kaggle/kagglehub#download-dataset).
